In [18]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
import chromadb
#from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

In [ ]:
from unstructured.partition.pdf import partition_pdf


def partition_document(file_path: str):
    """Extract text elements from PDF (simple RAG version)"""
    
    print(f"📄 Partitioning document: {file_path}")

    elements = partition_pdf(
        filename=file_path,
        strategy="fast",  # much faster than hi_res
    )

    print(f"✅ Extracted {len(elements)} elements")

    # convert elements to plain text
    texts = [el.text for el in elements if hasattr(el, "text") and el.text]

    return texts


# Test
file_path = "Attention_is_all_you_need.pdf"
texts = partition_document(file_path)

for t in texts[:50]:
    print("\n---\n", t)

📄 Partitioning document: Attention_is_all_you_need.pdf
✅ Extracted 393 elements

---
 3 2 0 2

---
 g u A 2

---
 ] L C . s c [

---
 7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a

---
 Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.


In [5]:
def create_chunks(texts):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,       # max characters per chunk
        chunk_overlap=100     # overlap for context continuity
    )

    chunks = text_splitter.split_text("\n".join(texts))

    print(f"✅ Created {len(chunks)} chunks")

    return chunks

chunks = create_chunks(texts)

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i} ---\n")
    print(chunk)

✅ Created 111 chunks

--- Chunk 0 ---

3 2 0 2
g u A 2
] L C . s c [
7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.
Attention Is All You Need
Ashish Vaswani∗ Google Brain avaswani@google.com
Noam Shazeer∗ Google Brain noam@google.com
Niki Parmar∗ Google Research nikip@google.com
Jakob Uszkoreit∗ Google Research usz@google.com
Llion Jones∗ Google Research llion@google.com

--- Chunk 1 ---

Jakob Uszkoreit∗ Google Research usz@google.com
Llion Jones∗ Google Research llion@google.com
Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu
Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com
Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com
Abstract

--- Chunk 2 ---

The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4775.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
client = chromadb.Client()

collection = client.create_collection(
    name="rag_collection"
)

In [9]:
def embed_and_store(chunks):

    embeddings = model.encode(chunks)

    ids = [f"id_{i}" for i in range(len(chunks))]

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids
    )

    print(f"✅ Stored {len(chunks)} chunks in ChromaDB")
    
embed_and_store(chunks)

✅ Stored 111 chunks in ChromaDB


In [10]:
query = "What is self attention?"

query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)

print(results["documents"])

[['Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22].', 'As side benefit, self-attention could yield more interpretable models. We inspect attention distributions from our models and present and discuss examples in the appendix. Not only do individual attention heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic and semantic structure of the sentences.\n5 Training\nThis section describes the training regime for our models.\n5.1 Training Data and Batching', '4 Why Self-Attention\nIn this section we compare various aspects of self-attention layers to the recurrent and convolu- tional layers commonly

In [12]:
results = collection.query(
    query_embeddings = query_embedding,
    n_results=5
)

docs = results["documents"][0]
metas = results["metadatas"][0]
distances = results["distances"][0]

for i, (doc, meta, dist) in enumerate(zip(docs, metas, distances), 1):
    similarity = dist   # convert distance → similarity
    
    print(f"\nResult {i}")
    print("-"*60)
    print("Similarity:", round(similarity, 3))
    print("Metadata:", meta)
    print("Text:", doc)


Result 1
------------------------------------------------------------
Similarity: 0.625
Metadata: None
Text: Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 27, 28, 22].

Result 2
------------------------------------------------------------
Similarity: 0.88
Metadata: None
Text: As side benefit, self-attention could yield more interpretable models. We inspect attention distributions from our models and present and discuss examples in the appendix. Not only do individual attention heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic and semantic structure of the sentences.
5 Training
This section describes th

In [13]:
queries = [
    "What datasets were used to train the model?",
    "What benchmark datasets were used for evaluation?",
    "What evaluation metrics were used?",
    "What model architecture or method is proposed?"
]

In [14]:
def retrieve_chunks(queries, k=2):

    retrieved_chunks = []

    for query in queries:

        query_embedding = model.encode([query])[0]

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=k
        )

        docs = results["documents"][0]

        retrieved_chunks.extend(docs)

    return retrieved_chunks

In [15]:
chunks = retrieve_chunks(queries, k=2)

context = "\n\n".join(chunks)

In [27]:
for chunk in chunks:
    print(chunk)
    print("______________________________________________________________________________________________________________________________")

all previously published models and ensembles, at a fraction of the training cost of any of the competitive models.
______________________________________________________________________________________________________________________________
We trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million sentence pairs. Sentences were encoded using byte-pair encoding [3], which has a shared source- target vocabulary of about 37000 tokens. For English-French, we used the significantly larger WMT 2014 English-French dataset consisting of 36M sentences and split tokens into a 32000 word-piece vocabulary [38]. Sentence pairs were batched together by approximate sequence length. Each training batch contained a
______________________________________________________________________________________________________________________________
Table 2 summarizes our results and compares our translation quality and training costs to other model architectures from the liter

In [22]:
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

In [24]:
prompt = f"""
You are analyzing an AI research paper.

Extract the following information from the provided context.

Return JSON in the following format:

{{
 "datasets_used": "...",
 "benchmark_dataset": "...",
 "evaluation_metric": "...",
 "method": "..."
}}

Rules:
- If the information is not present, return null.
- Only return JSON.
- Do not explain anything.

CONTEXT:
{context}
"""

In [25]:
response = llm.invoke(prompt)

print(response.content)

{
 "datasets_used": "WMT 2014 English-German dataset (about 4.5 million sentence pairs), WMT 2014 English-French dataset (36M sentences)",
 "benchmark_dataset": "WMT 2014 English-German translation task, WMT 2014 English-to-French translation task",
 "evaluation_metric": "BLEU score",
 "method": "Transformer architecture"
}
